<a href="https://colab.research.google.com/github/sunonmountain/High-Intent-Prospecting-Stack/blob/main/harvester.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import requests
import pandas as pd
from google.colab import userdata

SERPER_KEY = userdata.get("SERPER_API_KEY")

def multi_city_harvester(niche="SaaS", cities=["London", "Manchester", "Leeds"]):
    all_discovered_domains = []

    for city in cities:
        print(f"🕵️ Searching {niche} in {city}...")

        url = "https://google.serper.dev/search"
        query = f'site:.co.uk "{niche}" "{city}" -inurl:blog'

        payload = {"q": query, "num": 10}
        headers = {'X-API-KEY': SERPER_KEY, 'Content-Type': 'application/json'}

        try:
            response = requests.post(url, json=payload, headers=headers)
            results = response.json().get('organic', [])

            for result in results:
                link = result.get('link')
                domain = link.split('//')[-1].split('/')[0].replace('www.', '')

                all_discovered_domains.append({
                    "company_name": result.get('title'),
                    "domain": domain,
                    "city": city,
                    "niche": niche
                })
        except Exception as e:
            print(f"⚠️ Error in {city}: {e}")

    # Deduplicate: Ensure the same domain doesn't appear twice
    df = pd.DataFrame(all_discovered_domains)
    df.drop_duplicates(subset=['domain'], inplace=True)

    print(f"✅ Total Unique Domains Harvested: {len(df)}")
    return df

if __name__ == "__main__":
    target_cities = ["London", "Birmingham", "Glasgow"]
    results_df = multi_city_harvester(niche="Fintech", cities=target_cities)
    # Save for the next stage: People Discovery
    results_df.to_csv("harvested_companies.csv", index=False)

🕵️ Searching Fintech in London...
🕵️ Searching Fintech in Birmingham...
🕵️ Searching Fintech in Glasgow...
✅ Total Unique Domains Harvested: 23
